# ask

> Answer questions about a video course using LLM tool use over the yttoc cache

In [ ]:
#| default_exp ask

## Design

`yttoc-ask` is a read-only reasoning shell. It exposes 2 thin tools (`get_summaries`, `get_xscript_range`) to an LLM via the OpenAI Responses API. The LLM decides what to fetch. Python only formats output.

See `docs/superpowers/specs/2026-04-16-yttoc-ask-design.md` for full design rationale.

In [ ]:
#| export
import json, sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable
from pydantic import BaseModel, Field

In [ ]:
#| export
SYSTEM_PROMPT = """You are answering questions about a video course.
Use the provided tools to look up information. Do not fabricate content.
Answer in the user's language. If data is unavailable, say so."""

# --- Tool input schemas (Pydantic = single schema source) ---

class GetSummariesArgs(BaseModel):
    video_id: str = Field(description="YouTube video ID")

class GetXscriptRangeArgs(BaseModel):
    video_id: str = Field(description="YouTube video ID")
    start: float = Field(ge=0, description="Start time in seconds")
    end: float = Field(ge=0, description="End time in seconds")

# --- Final structured output ---

class Citation(BaseModel):
    video_id: str = Field(description="YouTube video ID")
    seconds: int = Field(ge=0, description="Timestamp in seconds")

class AskResponse(BaseModel):
    answer: str = Field(description="Synthesized answer to the user's question")
    citations: list[Citation] = Field(default_factory=list,
        description="Deep-link citations as {video_id, seconds}")

# --- Tool registry ---

@dataclass
class ToolEntry:
    """One tool in the registry — like a C struct bundling schema + handler."""
    schema: dict[str, Any]       # OpenAI function-call schema
    args_model: type[BaseModel]  # Pydantic model for arg validation
    handler: Callable[..., Any]  # The function to call

def make_tool(name: str, description: str,
              args_model: type[BaseModel],
              handler: Callable[..., Any]) -> ToolEntry:
    "Bundle a tool's schema, arg model, and handler into a ToolEntry."
    return ToolEntry(
        schema={
            'type': 'function',
            'name': name,
            'description': description,
            'parameters': {
                **args_model.model_json_schema(),
                'additionalProperties': False,
            },
        },
        args_model=args_model,
        handler=handler,
    )

def openai_tools(registry: dict[str, ToolEntry]) -> list[dict[str, Any]]:
    "Extract OpenAI tool schemas from a registry."
    return [tool.schema for tool in registry.values()]

def _to_jsonable(o):
    "Recursively convert Pydantic BaseModel instances to dicts for JSON serialization."
    if isinstance(o, BaseModel): return o.model_dump()
    if isinstance(o, list): return [_to_jsonable(x) for x in o]
    if isinstance(o, dict): return {k: _to_jsonable(v) for k, v in o.items()}
    return o

def dispatch_tool(registry: dict[str, ToolEntry], name: str, raw_args: str) -> str:
    "Validate args via Pydantic, call handler, return JSON result."
    tool = registry.get(name)
    if tool is None:
        return json.dumps({'error': f'Unknown tool: {name}'}, ensure_ascii=False)
    try:
        args = tool.args_model.model_validate_json(raw_args)
        result = tool.handler(**args.model_dump())
    except Exception as e:
        result = {'error': str(e)}
    try:
        return json.dumps(_to_jsonable(result), ensure_ascii=False)
    except (TypeError, ValueError) as e:
        return json.dumps({'error': f'Serialization failed: {e}'}, ensure_ascii=False)

## format_citations

In [ ]:
#| export
from yttoc.summarize import get_summaries as _read_summaries
from yttoc.xscript import get_xscript_range as _read_xscript_range
from yttoc.core import fmt_duration

def _find_section(sections: list[dict], seconds: int) -> dict | None:
    "Find the section containing the given timestamp. Returns None if no match."
    for s in sections:
        if s['start'] <= seconds < s['end']:
            return s
    return None

def format_citations(citations: list[Citation], # List of Citation objects
                     root: Path = None # Cache root for summaries lookup
                    ) -> list[str]: # Formatted citation lines
    "Resolve Citation objects into display lines with YouTube deep links."
    from yttoc.fetch import _DEFAULT_ROOT
    root = root or _DEFAULT_ROOT
    lines = []
    for i, c in enumerate(citations, 1):
        vid, sec = c.video_id, c.seconds
        url = f'https://youtu.be/{vid}?t={sec}'
        ts = fmt_duration(sec)
        sums = _read_summaries(vid, root)
        if 'error' in sums:
            lines.append(f'  [{i}] {vid} @ {ts}\n      {url}')
            continue
        title = sums['video'].get('title', vid)
        section = _find_section(sums['sections'], sec)
        if section:
            lines.append(f'  [{i}] {title} \u00a7{section["path"]} "{section["title"]}" @ {ts}\n      {url}')
        else:
            lines.append(f'  [{i}] {title} @ {ts}\n      {url}')
    return lines

def build_registry(root: Path) -> dict[str, ToolEntry]:
    "Build the tool registry with handlers bound to the given cache root."
    return {
        'get_summaries': make_tool(
            name='get_summaries',
            description='Return summaries.json verbatim for one cached video.',
            args_model=GetSummariesArgs,
            handler=lambda video_id: _read_summaries(video_id, root),
        ),
        'get_xscript_range': make_tool(
            name='get_xscript_range',
            description='Return raw parsed transcript segments within [start, end).',
            args_model=GetXscriptRangeArgs,
            handler=lambda video_id, start, end: _read_xscript_range(video_id, start, end, root),
        ),
    }

In [ ]:
# Test: format_citations resolves Citation objects to display string
from tempfile import TemporaryDirectory

with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID1'; v.mkdir()
    (v / 'summaries.json').write_text(json.dumps({
        'video': {'id': 'VID1', 'title': 'Test Video', 'channel': 'C',
                  'url': 'https://www.youtube.com/watch?v=VID1',
                  'duration': 600, 'upload_date': '20260101'},
        'sections': [
            {'path': '1', 'title': 'Intro', 'start': 0, 'end': 300,
             'summary': 's', 'keywords': [], 'evidence': {'text': '', 'at': 0}},
            {'path': '2', 'title': 'Main', 'start': 300, 'end': 600,
             'summary': 's', 'keywords': [], 'evidence': {'text': '', 'at': 300}},
        ],
        'full': {'summary': '', 'keywords': [], 'evidence': {'text': '', 'at': 0}},
    }))

    citations = [Citation(video_id='VID1', seconds=350)]
    lines = format_citations(citations, root)
    assert len(lines) == 1
    assert 'Test Video' in lines[0]
    assert '§2' in lines[0]
    assert '"Main"' in lines[0]
    assert 'https://youtu.be/VID1?t=350' in lines[0]
    assert '5:50' in lines[0]
print('ok')

In [ ]:
# Test: format_citations handles seconds not in any section
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID2'; v.mkdir()
    (v / 'summaries.json').write_text(json.dumps({
        'video': {'id': 'VID2', 'title': 'V2', 'channel': 'C',
                  'url': 'https://www.youtube.com/watch?v=VID2',
                  'duration': 100, 'upload_date': '20260101'},
        'sections': [
            {'path': '1', 'title': 'Only', 'start': 10, 'end': 50,
             'summary': 's', 'keywords': [], 'evidence': {'text': '', 'at': 10}},
        ],
        'full': {'summary': '', 'keywords': [], 'evidence': {'text': '', 'at': 0}},
    }))

    lines = format_citations([Citation(video_id='VID2', seconds=80)], root)
    assert len(lines) == 1
    assert 'https://youtu.be/VID2?t=80' in lines[0]
print('ok')

## Tool dispatcher and ask() loop

In [ ]:
# Test: dispatch_tool validates args via Pydantic and routes correctly
from tempfile import TemporaryDirectory

with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID_DT'; v.mkdir()
    (v / 'summaries.json').write_text(json.dumps({
        'video': {'id': 'VID_DT'}, 'sections': [], 'full': {}}))
    (v / 'captions.en.srt').write_text(
        '1\n00:00:00,000 --> 00:00:03,000\nhello\n')

    registry = build_registry(root)

    r1 = json.loads(dispatch_tool(registry, 'get_summaries', '{"video_id": "VID_DT"}'))
    assert r1['video']['id'] == 'VID_DT'

    r2 = json.loads(dispatch_tool(registry, 'get_xscript_range',
                                  '{"video_id": "VID_DT", "start": 0, "end": 10}'))
    assert isinstance(r2, list)

    r3 = json.loads(dispatch_tool(registry, 'get_summaries', '{"video_id": "NOPE"}'))
    assert 'error' in r3

    r4 = json.loads(dispatch_tool(registry, 'bad_tool', '{}'))
    assert 'error' in r4

    # Pydantic validation catches bad types
    r5 = json.loads(dispatch_tool(registry, 'get_xscript_range',
                                  '{"video_id": 123, "start": "bad", "end": 10}'))
    assert 'error' in r5
print('ok')

In [ ]:
# Test: _to_jsonable handles BaseModel, list[BaseModel], nested dict, passthrough
from yttoc.ask import _to_jsonable
from yttoc.core import Segment

s = Segment(start=1.0, end=2.0, text='hi')

# Single BaseModel → dict
assert _to_jsonable(s) == {'start': 1.0, 'end': 2.0, 'text': 'hi'}

# list[BaseModel] → list[dict]
assert _to_jsonable([s, s]) == [
    {'start': 1.0, 'end': 2.0, 'text': 'hi'},
    {'start': 1.0, 'end': 2.0, 'text': 'hi'},
]

# Nested: dict containing a BaseModel
assert _to_jsonable({'a': s, 'b': 1}) == {'a': {'start': 1.0, 'end': 2.0, 'text': 'hi'}, 'b': 1}

# Nested: dict containing list[BaseModel]
assert _to_jsonable({'items': [s]}) == {'items': [{'start': 1.0, 'end': 2.0, 'text': 'hi'}]}

# Passthrough for scalars
assert _to_jsonable(42) == 42
assert _to_jsonable('abc') == 'abc'
assert _to_jsonable(None) is None

# Passthrough for plain dict / list (idempotent)
assert _to_jsonable({'x': 1, 'y': [2, 3]}) == {'x': 1, 'y': [2, 3]}

print('ok')

In [ ]:
# Test: dispatch_tool serializes list[Segment] handler result as list of {start,end,text} dicts
import json
from tempfile import TemporaryDirectory
from pathlib import Path
from yttoc.ask import dispatch_tool, build_registry

with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID_BND'; v.mkdir()
    (v / 'captions.en.srt').write_text(
        '1\n00:00:00,000 --> 00:00:03,000\nalpha\n\n'
        '2\n00:00:05,000 --> 00:00:08,000\nbeta\n')

    registry = build_registry(root)
    raw = dispatch_tool(registry, 'get_xscript_range',
                        '{"video_id":"VID_BND","start":0,"end":10}')
    parsed = json.loads(raw)
    assert isinstance(parsed, list), f'expected list, got {type(parsed)}'
    assert len(parsed) == 2
    for item in parsed:
        assert isinstance(item, dict)
        assert set(item.keys()) == {'start', 'end', 'text'}
        assert isinstance(item['start'], float)
        assert isinstance(item['end'], float)
        assert isinstance(item['text'], str)
print('ok')

In [ ]:
#| export
import openai

def ask(question: str, # Natural-language query
        video_ids: list[str], # Cached video IDs
        model: str = 'gpt-4o', # LLM model
        max_iterations: int = 20, # Safety cap on tool-use loop
        root: Path = None, # Cache root directory
        verbose: bool = False # Print tool calls to stderr
       ) -> AskResponse:
    "Run a tool-use loop to answer a question about a video course."
    from yttoc.fetch import _DEFAULT_ROOT
    root = root or _DEFAULT_ROOT
    client = openai.OpenAI()
    registry = build_registry(root)
    tools = openai_tools(registry)

    user_input = f"Video IDs: {json.dumps(video_ids)}\n---\nQuestion: {question}"

    response = client.responses.parse(
        model=model,
        instructions=SYSTEM_PROMPT,
        input=user_input,
        tools=tools,
        text_format=AskResponse,
    )

    for _ in range(max_iterations):
        tool_calls = [item for item in response.output
                      if item.type == 'function_call']
        if not tool_calls:
            break

        tool_outputs = []
        for tc in tool_calls:
            if verbose:
                args = json.loads(tc.arguments)
                print(f'{tc.name}({", ".join(f"{k}={v!r}" for k,v in args.items())})',
                      file=sys.stderr)
            result = dispatch_tool(registry, tc.name, tc.arguments)
            tool_outputs.append({
                'type': 'function_call_output',
                'call_id': tc.call_id,
                'output': result,
            })

        response = client.responses.parse(
            model=model,
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools,
            text_format=AskResponse,
        )

    if hasattr(response, 'output_parsed') and response.output_parsed is not None:
        return response.output_parsed

    return AskResponse(answer='No answer generated.', citations=[])

In [ ]:
#| eval: false
# Integration test: ask() returns AskResponse with a real cached video
from yttoc.fetch import _DEFAULT_ROOT

result = ask('What is this course about?', ['7T83srD0Mu4'], root=_DEFAULT_ROOT)
assert isinstance(result, AskResponse)
print(f"Answer: {result.answer[:200]}")
print(f"Citations: {result.citations}")

## CLI

In [ ]:
#| export
from fastcore.script import call_parse, Param

@call_parse
def yttoc_ask(question: str, # Natural-language query
              ids: Param('Cached video IDs (1+ required)', str, nargs='+'), # Video IDs
              model: str = 'gpt-4o', # LLM model
              max_iterations: int = 20, # Safety cap on tool-use loop
              root: str = None, # Root cache directory
              verbose: bool = False, # Show tool-use trace on stderr
             ):
    "Answer a question about a video course using LLM tool use."
    from yttoc.fetch import _DEFAULT_ROOT
    root_path = Path(root) if root else _DEFAULT_ROOT

    result = ask(question, ids, model=model, max_iterations=max_iterations,
                 root=root_path, verbose=verbose)

    print(result.answer)

    if result.citations:
        lines = format_citations(result.citations, root_path)
        print()
        print('Citations:')
        for line in lines:
            print(line)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()